# Other tasks with Transformers

Here we discuss about document (sequence) first, and then token later

# For Sentence / Sequence

## Text summary

### Abstractive Summarization
Generates a new, concise summary by rephrasing the original content rather than copying text verbatim. The model synthesizes the main points, often resulting in a more natural and coherent summary.

### Extractive Summarization
Selects key sentences or phrases directly from the original text to form a summary. It maintains the exact wording from the source, which often preserves factual details and important context.

In [ ]:
#!pip install transformers bert-extractive-summarizer evaluate nltk rouge_score datasets

In [ ]:
import nltk
nltk.download('punkt')  # Download NLTK tokenizer models

from transformers import pipeline
from summarizer import Summarizer  # from bert-extractive-summarizer
import evaluate

# ================================
# Sample document and reference summary
# ================================
document = (
    "The field of artificial intelligence has advanced rapidly over the last decade. "
    "Researchers have developed numerous models that excel in tasks like language understanding, "
    "vision recognition, and robotics. These advancements have led to unprecedented innovations in technology, "
    "which in turn have impacted industries such as healthcare, finance, and transportation. "
    "The growth in computational power and the availability of large datasets have further accelerated the pace of this research. "
    "Many believe that AI will continue to transform the world in the coming years."
)

# A reference summary (for evaluation purposes)
reference_summary = (
    "Artificial intelligence has seen rapid advancements, impacting various industries including healthcare and finance, "
    "driven by increased computational power and data availability."
)

# ================================
# 1. Abstractive Summarization using a Transformers Pipeline
# ================================
print("=== Abstractive Summarization ===")
abstractive_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

# Generate abstractive summary.
abstractive_summary = abstractive_summarizer(
    document, max_length=130, min_length=30, do_sample=False
)[0]['summary_text']

print("Abstractive Summary:")
print(abstractive_summary)

# ================================
# 2. Extractive Summarization using bert-extractive-summarizer
# ================================
print("\n=== Extractive Summarization ===")
# The Summarizer from bert-extractive-summarizer returns a summary based on the input ratios.
extractive_model = Summarizer(model="distilbert-base-uncased")
extractive_summary = extractive_model(document, ratio=0.3)  # Adjust ratio as needed

print("Extractive Summary:")
print(extractive_summary)




In [ ]:
# install if needed:
# pip install rouge-score

from rouge_score import rouge_scorer, scoring

# 1. Initialize a scorer for the ROUGE types you want
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],    # can add "rougeLsum" or "rougeS*" if desired
    use_stemmer=True
)

# 2. Score your summaries
scores_abs = scorer.score(reference_summary, abstractive_summary)
scores_ext = scorer.score(reference_summary, extractive_summary)

# 3. Print precision, recall, f1 for each ROUGE type
print("=== Abstractive ===")
for key, result in scores_abs.items():
    print(f"{key}: P={result.precision:.3f}, R={result.recall:.3f}, F1={result.fmeasure:.3f}")

print("\n=== Extractive ===")
for key, result in scores_ext.items():
    print(f"{key}: P={result.precision:.3f}, R={result.recall:.3f}, F1={result.fmeasure:.3f}")


The ROUGE (Recall-Oriented Understudy for Gisting Evaluation) metric is a set of measures used to evaluate the quality of automatically generated summaries (or translations) by comparing them to reference (human‐written) summaries. It focuses on overlap of units such as n-grams, longest common subsequences, and skip-grams.

ROUGE-1: unigram overlap

ROUGE-2: bigram overlap

ROGUE-L: Based on the Longest Common Subsequence (LCS). Captures sentence‐level structure similarity.


## Question Answering with transformers

In [ ]:
# Install transformers if you haven't already:
# pip install transformers

from transformers import pipeline

# Create a question answering pipeline using a pre-trained model
qa_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")

# Define a sample context or passage
context = """
The Hugging Face Transformers library provides state-of-the-art machine learning models for natural language processing tasks.
One of these tasks is question answering, where the model extracts an answer to a given question based on a provided context.
This approach is effective for answering questions based on documents, articles, and other forms of text.
"""

# Define your question
question = "What task is the Transformers library used for in the context?"

# Run the question answering pipeline
result = qa_pipeline(question=question, context=context)

# Print out the answer and its confidence score
print("Answer:", result["answer"])
print("Score:", result["score"])


## No shot and Few-Shot for sequence classification





### Zero-Shot Classification

Enables a model to classify text into categories without any prior task-specific training. The model predicts labels based solely on their descriptions.

In [ ]:
# --------------------------
# Zero-Shot Classification
# --------------------------
from transformers import pipeline

# Create a zero-shot classification pipeline using a model fine-tuned on MNLI.
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Example text to classify and candidate labels.
sequence_to_classify = "The product is great, but the price is too high."
candidate_labels = ["positive", "negative", "neutral"]

# Get the zero-shot classification result.
zero_shot_result = zero_shot_classifier(sequence_to_classify, candidate_labels=candidate_labels)
print("Zero-Shot Classification Result:")
print(zero_shot_result)





### Few-Shot Classification
Involves providing a model with a small number of labeled examples (few shots) in a prompt so that it can adapt its output to classify new instances. The model leverages these examples to understand the task, even if it wasn’t extensively fine-tuned for that specific classification.

In [ ]:
# --------------------------
# Few-Shot Classification
# --------------------------
from transformers import pipeline

# Here we use a generative model (e.g., GPT-Neo) to perform few-shot classification
# by including a few examples in the prompt.
few_shot_classifier = pipeline("text-generation", model="EleutherAI/gpt-neo-125M")

# Prepare a few-shot prompt with a few examples of sentiment classification.
prompt = """
Classify the sentiment of the following sentence as positive, negative, or neutral.
Example 1:
Sentence: "The movie was fantastic and the story was compelling."
Sentiment: positive

Example 2:
Sentence: "I was extremely disappointed with the service."
Sentiment: negative

Example 3:
Sentence: "The product was okay and nothing special happened."
Sentiment: neutral

Now classify the following sentence:
Sentence: "The food was delicious but the service was slow."
Sentiment:"""

# Generate a completion; note that the output might include additional text.
few_shot_result = few_shot_classifier(prompt, max_length=150, do_sample=False)

print("\nFew-Shot Classification Result:")
print(few_shot_result[0]['generated_text'])

In [ ]:
from  datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer # Import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, and Trainer

# -- Create a larger dataset for traditional classification --
# This is just illustrative; in practice, you would load a full dataset.
data = {
    "text": [
        "I love this movie!",
        "Absolutely fantastic experience.",
        "I didn't like the film.",
        "The movie was terrible.",
        "An unforgettable film with brilliant acting.",
        "The film was a waste of time.",
        "A masterpiece of cinematography.",
        "I would not recommend this movie to anyone.",
        "The storyline was engaging and well developed.",
        "Poor performance by the lead actor."
    ],
    "label": [1, 1, 0, 0, 1, 0, 1, 0, 1, 0]  # 1: positive, 0: negative
}
trad_dataset = Dataset.from_dict(data)

# -- Load the pre-trained model and tokenizer --
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# -- Tokenize the dataset --
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = trad_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")

# -- Define training arguments for traditional training --
training_args = TrainingArguments(
    output_dir="./traditional_model",
    num_train_epochs=3,                # Fewer epochs because we have more data
    per_device_train_batch_size=4,     # Larger batch size now, for stability
    learning_rate=5e-5,
    logging_steps=1,
    report_to="none"  # disables wandb
)

# -- Create the Trainer and train --
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)
trainer.train()

# -- Inference --
texts = ["What a fantastic movie!", "I really hated that film."]
inputs = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors="pt")
outputs = model(**inputs)
import numpy as np
predictions = np.argmax(outputs.logits.detach().numpy(), axis=1)
print("Traditional Classification Predictions (0: negative, 1: positive):", predictions)


# For token/ NER

## Question answering using NER via Spacy

In [ ]:
# Question Answering using Spacy
import spacy

def answer_question_using_ner(question, context):
    # Load spaCy's small English model.
    nlp = spacy.load("en_core_web_sm")

    # Process the context to obtain a spaCy Doc.
    doc = nlp(context)

    # For demonstration, we use a simple heuristic:
    # If the question starts with "Who", we assume we need to look for PERSON entities.
    if question.strip().lower().startswith("who"):
        # Tokenize the question and remove stop words and punctuation
        question_doc = nlp(question)
        keywords = [token.text for token in question_doc if not token.is_stop and token.is_alpha]

        # Iterate through each sentence in the context.
        for sent in doc.sents:
            # Check if the sentence contains at least one keyword from the question.
            # (A more refined version could use similarity or more advanced matching.)
            if any(keyword.lower() in sent.text.lower() for keyword in keywords):
                # Search for PERSON entities in that sentence.
                for ent in sent.ents:
                    if ent.label_ == "PERSON":
                        return ent.text  # Return the first PERSON found.
    # For other question types, more rules would be needed.
    return "No answer found."

# --- Example usage ---
context = (
    "John Smith, the CEO of OpenAI, visited San Francisco last week. "
    "He met with representatives from various tech companies in California."
)
question = "Who is the CEO of OpenAI?"

answer = answer_question_using_ner(question, context)
print("Question:", question)
print("Answer:", answer)


## Question answering using NER + transformers

In [ ]:
from transformers import pipeline

# --- Step 1: Create the pipelines ---
# QA pipeline (for finding answer spans)
qa_pipeline = pipeline("question-answering", model="distilbert-base-uncased-distilled-squad")

# NER pipeline (for extracting entities)
# Use an NER model and set aggregation_strategy to combine B- and I- tags.
ner_pipeline = pipeline("ner", model="dbmdz/bert-large-cased-finetuned-conll03-english", aggregation_strategy="simple")

# --- Step 2: Define the context and question ---
context = (
    "John Smith, the CEO of OpenAI, visited San Francisco last week. "
    "He met with representatives from various tech companies in California."
)
question = "Who is the CEO of OpenAI?"

# --- Step 3: Run Question Answering ---
qa_result = qa_pipeline(question=question, context=context)
print("QA Answer:", qa_result["answer"])
print(f"Answer character span: [{qa_result['start']}:{qa_result['end']}]")

# --- Step 4: Run Named Entity Recognition ---
ner_results = ner_pipeline(context)
print("\nNER Entities:")
for ent in ner_results:
    print(f"{ent['entity_group']}: '{ent['word']}' (Span: {ent['start']}:{ent['end']}, Score: {ent['score']:.2f})")

# --- Step 5: Find overlap between QA answer and NER entities ---
# Use the character offsets from the QA result to check if any NER entity encloses the answer.
qa_start, qa_end = qa_result["start"], qa_result["end"]
matched_entity = None
for ent in ner_results:
    # Check if the NER entity spans the QA answer.
    if ent["start"] <= qa_start and ent["end"] >= qa_end:
        matched_entity = ent
        break

if matched_entity:
    print("\nQA Answer overlaps with NER entity:")
    print(f"Entity: '{matched_entity['word']}', Label: {matched_entity['entity_group']}")
else:
    print("\nQA Answer did not overlap with any detected NER entity.")


## Zero Shot and Few-Shot NER

there is no Zero Shot and Few-Shot  NER models except for using prompting

### Zero Shot NER
it's a prompt!

In [ ]:
#prompting for Zero Shot NER
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----- 1. Load a generative model capable of following instructions -----
model_name = "google/flan-t5-small"  # a model tuned for instruction following
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ----- 2. Prepare your text and a prompt for NER -----
text = "John Smith, the CEO of OpenAI, visited San Francisco last week."
prompt = f"Extract all named entities from the following text: {text}\nEntities: "

# ----- 3. Encode and generate the output -----
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model.generate(input_ids, max_length=64, num_beams=5, early_stopping=True)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Zero-Shot NER Output:")
print(result)


###Few-Shot NER
very similar, you provide an instruction and examples

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----- 1. Load an instruction-tuned model (e.g., FLAN-T5) -----
model_name = "google/flan-t5-small"  # You can also try larger variants (flan-t5-base, etc.)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ----- 2. Define a prompt with few-shot examples for NER -----
prompt = (
    "Extract all named entities from the following text and label them with their entity type. \n\n"
    "Example 1:\n"
    "Text: \"John Smith is the CEO of OpenAI.\"\n"
    "Output: John Smith [PERSON]; OpenAI [ORG]\n\n"
    "Example 2:\n"
    "Text: \"Alice works at Microsoft in Redmond.\"\n"
    "Output: Alice [PERSON]; Microsoft [ORG]; Redmond [LOC]\n\n"
    "Now, extract the named entities from the following text:\n"
    "Text: \" Ace teaches at Loyola Marymount University.\"\n"
    "Output:"
)

# ----- 3. Encode the prompt and generate the output -----
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model.generate(input_ids, max_length=200, num_beams=5, early_stopping=True)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# ----- 4. Print the output -----
print("Few-Shot NER Prompting Output:")
print(result)


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----- 1. Load an instruction-tuned model -----
model_name = "google/flan-t5-small"  # Try a larger variant if available (e.g., flan-t5-base)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ----- 2. Define the entity labels -----
labels = ["AGE",  "PATIENT" ]

# ----- 3. Construct a refined prompt with detailed output instructions -----
prompt = (
    "Extract and label all named entities from the following text.\n"
    "For each entity found, output it in the following format with the following entities:\n"
    .join(labels) + ".\n\n"
    "Example:\n"
    "Text: \"John Doe, a 55-year-old man, was treated by Dr. Smith at Mercy Hospital in Los Angeles, CA, on April 10, 2021.\"\n"
    "Output: \"John Doe [PATIENT]; 55 [AGE]; Dr. Smith [DOCTOR]; Mercy Hospital [HOSPITAL]; Los Angeles [CITY]; CA [STATE]; April 10, 2021 [DATE]\"\n\n"

    "Now, extract the entities from the text below:\n"
    "Text: \"Emily Davis, a 34-year-old woman, Dr. Michael Johnson cares with her, at CarePlus Clinic, "
    "located at 456 Elm Street, NewYork, NY has recommended starting insulin therapy. "
    "She has an appointment scheduled for March 15, 2024.\"\n"
)

# ----- 4. Encode the prompt and generate the output -----
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
outputs = model.generate(input_ids, max_length=256, num_beams=5, early_stopping=True)
result = tokenizer.decode(outputs[0], skip_special_tokens=True)

# ----- 5. Print the refined output -----
print("Zero-Shot (Few-Shot Prompted) NER Output:")
print(result)
